In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

In [ ]:
from marine_qc import (
    combine_qc_results,
    do_climatology_check,
    do_datetime_check,
    do_hard_limit_check,
    do_maritime_check,
    do_multiple_individual_check,
    do_position_check,
    do_valid_value_check,
    do_wind_consistency_check,
    plot_latitude_longitude,
)
from marine_qc.helpers.external_clim import get_climatological_value

In [ ]:
from data import get_climatology_data, get_individual_data

# How to use quality control checks with individual reports

The ``marine_qc`` toolbox provides some quality control (QC) checks that work on individual marine reports. Creating some tests dataset we can show how to use them on "real" data. There are several checks that are shown here:

* a position check that tests whether the latitude-longitude position of the report is valid
* a datetime check that tests whether both the date and the time of the report is valid
* a valid-value check that tests whether an observational value is present
* a hard-limit check that tests whether an observational value is within a specified range
* a wind-consistency check that tests whether wind speed and wind direction are consistent
* a maritime check that tests whether latitude-longitude is on sea whereby an external land-sea mask is needed
* a climatology check that tests whether an observational value is within an acceptable range around the mean of an external climatology

Finally, we run all these QC checks within a single function and combine the results of each QC check into a single QC flag.

For more information of all available QC checks on individual reports see [the overview of QC functions for individual reports](https://marine-qc.readthedocs.io/en/latest/overview_ind.html).

The test dataset we provide is a ``pandas.DataFrame`` including several individual marine reports. The QC functions return a QC flag for each individual report. The flags are:

* `0`: the check has passed
* `1`: the check has failed
* `2`: the check is untestable

In [ ]:
data = get_individual_data()
data

We have nine different marine reports that include six different parameters (`lat`, `lon`, `date`, `sea_surface_temperature`, `wind_speed` and `wind_direction`).

## Checks that need no external data

We start with the position check. The function needs latitude and longitude values as input.

In [ ]:
qc_pos = do_position_check(
    lat=data.lat,
    lon=data.lon,
)
pd.DataFrame({"location": data.location, "lat": data.lat, "lon": data.lon, "qc_pos": qc_pos})

All checks have passed except the check for the South Pacific Ocean. It seems that latitude and longitude values are swapped. These are the expected results.

The next check is the datetime check which only needs date values as input.

In [ ]:
qc_datetime = do_datetime_check(
    data.date,
)
pd.DataFrame({"location": data.location, "date": data.date, "qc_datetime": qc_datetime})

As expected all checks have passed except the check for the Gulf of Mexico. We get a `2` that stands for untestable since the datetime is not wrong but not available.

In the next few checks we concentrate on `sea_surface_temperature`. First of all, we want to check if the values are valid.

In [ ]:
qc_valid = do_valid_value_check(
    data.sea_surface_temperature,
)
pd.DataFrame({"location": data.location, "sea_surface_temperature": data.sea_surface_temperature, "qc_valid": qc_valid})

Since no values for Paris, Tokyo and Sydney are provided the checks fails for these reports as expected.

Next, we check whether the values are within a range between `10` and `30` `°C`.

In [ ]:
qc_hard = do_hard_limit_check(
    data.sea_surface_temperature,
    limits=(10, 30),
)
pd.DataFrame({"location": data.location, "sea_surface_temperature": data.sea_surface_temperature, "qc_hard": qc_hard})

As expected, the check fails for the Norwegian Sea since the value is under `10 °C` (`8.5 °C`). Interstingly, we get three `2`s (untestable) for locations where no values are provided. The aim of the hard-limit test is not to test whether a value is valid but to test whether a value is within a specified range. Therefore, we decided to return `2`s instead of `1`s (failed).

The last check that only need inputs from the dataset is the wind-consistency check. This is a so-called cross check that test whether two observsational values are consistent. In this check, `wind_speed` and `wind_direction` are consistent if zero `wind_speed` corresponds to no particular `wind_direction` and `wind_speed` above a threshold corresponds to a particular `wind_direction`.

In [ ]:
qc_wind = do_wind_consistency_check(
    wind_speed=data.wind_speed,
    wind_direction=data.wind_direction,
)
pd.DataFrame({"location": data.location, "wind_speed": data.wind_speed, "wind_direction": data.wind_direction, "qc_wind": qc_wind})

All checks have passed except the check for the Norwegian Sea. Here, zero `wind_speed` correspond to a particular `wind_direction` (`270°`).

## Checks that do need external data

Now, we focus on QC checks that need external data in addition to the individual reports. Firtly, we import some external data. This is a `xarray.Dataset` containing [CF](https://cfconventions.org/)-compliant data:

* a land-sea mask where `1` denotes a land point and `0` denotes a sea point (`land_sea_mask`)
* a simple, latitudinally and longitudinally dependent sea-surface temperature field (`sst`)
* a sea-surface temperature standard deviation of grid cell minus neighbourhood mean (`sst_stde1`)
* a sea-surface temperature standard deviation of point observation minus grid cell mean (`sst_stde2`)
* a sea-surface temperature standard deviation of neighbourhood mean uncertainty (`sst_stde3`)

In [ ]:
climatology_data = get_climatology_data()
climatology_data

Firstly, we check whether the positions of the marine reports are on sea using the external land-sea mask.

In [ ]:
climatology_data.land_sea_mask.plot()

We do not need to extract the latitude-longitude points amnually from the land-sea mask. This is done internally in the maritime check. We simply can pass the entire land-sea mask as a `xarray.DataArray` to the function.

In [ ]:
qc_sea = do_maritime_check(
    lat=data.lat,
    lon=data.lon,
    sea_land_mask=climatology_data.land_sea_mask,
    sea_flag=0,
)
pd.DataFrame({"location": data.location, "lat": data.lat, "lon": data.lon, "qc_sea": qc_sea})

As expected all checks have passed except for the land points (Paris, Tokyo and Sydney).

Finally, we check `sea_surface_temeprature` against the sea-surface climatology. The check should pass if the value from the report and the climatological value do not differ more than `5K`. As in the check above, we can provide the external data as `xarray.DataArray`.

In [ ]:
climatology_data.sst.plot()

We want to extract and show the climatological values that correspond to the latitude-longitude positions in the marine reports. Therefore, we can use the use the helper function `get_climatological_value` from `marine_qc.helpers.external_clim`.

In [ ]:
clim_sst = get_climatological_value(climatology_data.sst, lat=data.lat, lon=data.lon)

In [ ]:
qc_clim = do_climatology_check(
    value=data.sea_surface_temperature,
    climatology=climatology_data.sst,
    maximum_anomaly=5.0,
    lat=data.lat,
    lon=data.lon,
)
pd.DataFrame(
    {
        "location": data.location,
        "lat": data.lat,
        "lon": data.lon,
        "sea_surface_temperature": data.sea_surface_temperature,
        "climatology": clim_sst,
        "qc_clim": qc_clim,
    }
)

As for the hard-limit test, we get `2`s (untestable) for reports without sea-surface temperatures. The check for the Gulf of Mexico fails the the temperature difference is more than `5K`. All other checks have passed as expected.

## Run all checks within one function

We can run multiple QC checks within one function (`do_multiple_individual_check`). Therefore, we need a `pandas.DataFrame` and a nested QC dictionary. This is the structure of the dictionary:

* arbitrary user-specified name for the checks
  * "func": name of the QC function as `str` (mandatory)
  * "names": dictionary containing parameter names of the QC function and their corresponding columns in the `pandas.DataFrame` (mandatory)
  * "arguments": dictionary containing any key-word arguments passed to the QC function (optionally)
* ...

We define the QC dictionary according to the QC checks above.

In [ ]:
qc_dict = {
    "positional_check": {
        "func": "do_position_check",
        "names": {
            "lat": "lat",
            "lon": "lon",
        },
    },
    "datetime_check": {
        "func": "do_datetime_check",
        "names": {"date": "date"},
    },
    "hard_limit_check": {"func": "do_hard_limit_check", "names": {"value": "sea_surface_temperature"}, "arguments": {"limits": (10, 29)}},
    "maritime_check": {
        "func": "do_maritime_check",
        "names": {
            "lat": "lat",
            "lon": "lon",
        },
        "arguments": {
            "sea_land_mask": climatology_data.land_sea_mask,
            "sea_flag": 0,
        },
    },
    "climatology_check": {
        "func": "do_climatology_check",
        "names": {"value": "sea_surface_temperature"},
        "arguments": {
            "climatology": climatology_data.sst,
            "maximum_anomaly": 5.0,
            "lat": data.lat,
            "lon": data.lon,
        },
    },
    "wind_consistency_check": {
        "func": "do_wind_consistency_check",
        "names": {"wind_speed": "wind_speed", "wind_direction": "wind_direction"},
    },
}

Now, run all QC checks calling one function. We set `return_method` to "failed" which means that all requested QC check are run until the first check fails. The other QC checks are flagged as unstested (`3`).

In [ ]:
qc_multi = do_multiple_individual_check(
    data,
    qc_dict,
    return_method="failed",
)
qc_multi

Now, we combine the results into one final QC flag. The QC flag values are prioritized in this order: [`1`, `0`, `3`, `2`].

In [ ]:
qc_flag = combine_qc_results(qc_multi)
pd.DataFrame({**data, "qc_flag": qc_flag})

This is our final QC results. Let's make some plots.

In [ ]:
plot = plot_latitude_longitude(data.lat, data.lon, qc_flag, marker_size=15, add_coastlines=True)